# Ders 11: Dayanıklılık, Düşmanca Örnekler ve Dağılım Kayması

**İleri Derin Öğrenme** — Haydar Kılıç

Ön koşul: *Derin Öğrenme*, Ders 5 (Düzenlileştirme, genelleme).

Test doğruluğu, eğitim verisiyle aynı dağılımdan gelen veri üzerindeki performansı ölçer. Gerçek
kullanımda bunu sağlayan neredeyse hiçbir şey yoktur. Bu defterde bunun bozulduğu üç yolu ele
alıyoruz: bir rakip tarafından seçilen **düşmanca** bozulmalar, ortalamada geçerli ama gruplar
düzeyinde geçersiz olan **sahte korelasyonlar** ve eğitim ile dağıtım arasındaki **dağılım kayması**.
Her şey küçük sentetik modeller üzerinde inceleniyor; böylece mekanizmalar büyük bir ağın arkasına
gizlenmek yerine görünür oluyor.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(0)
plt.rcParams["figure.dpi"] = 100
sigmoid = lambda z: 1/(1+np.exp(-z))
print("Kütüphaneler yüklendi.")


## 1. Düşmanca Örnekler Doğrusallığın ve Boyutun Sonucudur

Özgün sezgi (Goodfellow vd.) derin bir ağa hiç ihtiyaç duymaz. $w^\top x$ doğrusal modeli için
$\lVert \delta\rVert_\infty \le \epsilon$ kısıtı altındaki en kötü bozulma
$\delta = \epsilon\,\text{sign}(w)$'dır ve logiti şu kadar değiştirir:

$$w^\top \delta = \epsilon \lVert w \rVert_1 \approx \epsilon\, d\, \mathbb{E}|w_i| .$$

Yani her piksel fark edilemez bir $\epsilon$ kadar hareket ederken değişim **boyutla doğrusal**
büyür. Dolayısıyla düşmanca örnekler egzotik bir patolojinin kanıtı değildir; neredeyse doğrusal bir
fonksiyonun çok yüksek boyutlu bir uzayda değerlendirilmesinin doğal sonucudur.


In [ ]:
dims = np.array([10, 50, 100, 500, 1000, 5000, 10000])
eps = 0.01
rng = np.random.default_rng(0)
shift = [eps*np.abs(rng.normal(size=d)/np.sqrt(d)).sum() for d in dims]     # ||w||_2 = 1 iken ||w||_1

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].loglog(dims, shift, "o-", lw=2)
axes[0].set_xlabel("girdi boyutu d"); axes[0].set_ylabel("eps=0.01 saldırısında logit değişimi")
axes[0].set_title("Piksel başına bozulma bütçesi sabit; etki sqrt(d) ile büyüyor")
axes[0].grid(alpha=0.3, which="both")

# Saldırının geometrik olarak ne yaptığının 2B görünümü
w = np.array([1.0, 0.4]); w /= np.linalg.norm(w)
pts = rng.normal(size=(300, 2))
y = (pts@w > 0)
eps2 = 0.6
adv = pts - eps2*np.sign(w)*np.where(y, 1, -1)[:, None]
axes[1].scatter(pts[y, 0], pts[y, 1], s=10, c="steelblue", alpha=0.6, label="sınıf +")
axes[1].scatter(pts[~y, 0], pts[~y, 1], s=10, c="orange", alpha=0.6, label="sınıf -")
xs = np.linspace(-3, 3, 10)
axes[1].plot(xs, -w[0]/w[1]*xs, "k--", lw=1.5, label="karar sınırı")
for i in range(0, 300, 12):
    axes[1].annotate("", xy=adv[i], xytext=pts[i],
                     arrowprops=dict(arrowstyle="->", color="crimson", lw=0.8))
axes[1].set_xlim(-3, 3); axes[1].set_ylim(-3, 3); axes[1].legend(fontsize=8)
axes[1].set_title("L-sonsuz saldırısı her koordinatı sınıra doğru eps kadar kaydırır")

# Doğrusal bir model için, çeşitli boyutlarda doğruluk-epsilon eğrisi
def linear_acc_under_attack(d, eps_list, n=2000, seed=0):
    r = np.random.default_rng(seed)
    w = r.normal(size=d)/np.sqrt(d)
    X = r.normal(size=(n, d)); yv = np.sign(X@w)
    margins = np.abs(X@w)
    return [np.mean(margins > e*np.abs(w).sum()) for e in eps_list]

eps_list = np.linspace(0, 0.2, 40)
for d, c in zip([10, 100, 1000], ["#c6dbef", "#6baed6", "#08519c"]):
    axes[2].plot(eps_list, linear_acc_under_attack(d, eps_list), lw=2, c=c, label=f"d={d}")
axes[2].set_xlabel("bozulma bütçesi eps (L-sonsuz)"); axes[2].set_ylabel("saldırı altında doğruluk")
axes[2].set_title("Daha yüksek boyut = daha ucuz saldırı"); axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 2. Bir Optimizasyon Problemi Olarak FGSM ve PGD

Düşmanca değerlendirme kısıtlı bir maksimizasyondur:

$$\max_{\lVert\delta\rVert_p \le \epsilon} \ \mathcal{L}\big(f_\theta(x+\delta),\ y\big).$$

**FGSM** tek bir işaretli gradyan adımı atar: hızlıdır ama doğrusal olmayan bir kaybın doğrusal
yaklaşımıdır. **PGD** ise yineler — küçük adımlar, her seferinde $\epsilon$-yuvarına geri izdüşüm ve
kötü bir yerel optimumdan kaçmak için rastgele başlangıçlar. PGD'nin standart *değerlendirme* saldırısı
olmasının nedeni tam olarak şudur: zayıf saldırılar yanlış bir güven üretir. FGSM'i atlatan ama PGD'ye
yenilen bir savunma aslında hiç dayanıklı değildi; yalnızca gradyanı gizlemişti.


In [ ]:
# Sentetik bir ikili görevde eğitilen küçük 2 katmanlı bir ağ
rng = np.random.default_rng(1)
d, n = 20, 1500
w_star = rng.normal(size=d)
X = rng.normal(size=(n, d)); Y = (X@w_star > 0).astype(float)

def init(seed=0, h=32):
    r = np.random.default_rng(seed)
    return [r.normal(size=(d, h))/np.sqrt(d), np.zeros(h), r.normal(size=h)/np.sqrt(h), 0.0]

def forward(p, X):
    W1, b1, w2, b2 = p
    A = np.maximum(X@W1 + b1, 0)
    return A@w2 + b2, A

def loss_grads(p, X, Y):
    W1, b1, w2, b2 = p
    z, A = forward(p, X)
    g = (sigmoid(z) - Y)/len(X)
    gw2 = A.T@g; gb2 = g.sum()
    gA = np.outer(g, w2)*(A > 0)
    return [X.T@gA, gA.sum(0), gw2, gb2]

def input_grad(p, X, Y):
    W1, b1, w2, b2 = p
    z, A = forward(p, X)
    g = (sigmoid(z) - Y)
    return (np.outer(g, w2)*(A > 0)) @ W1.T

def train(p, X, Y, steps=1500, lr=0.5, adv_eps=0.0, pgd_steps=7):
    for _ in range(steps):
        Xb = X
        if adv_eps > 0:                                    # düşmanca eğitim
            Xb = pgd(p, X, Y, adv_eps, pgd_steps)
        gs = loss_grads(p, Xb, Y)
        p = [q - lr*g for q, g in zip(p, gs)]
    return p

def fgsm(p, X, Y, eps):
    return X + eps*np.sign(input_grad(p, X, Y))

def pgd(p, X, Y, eps, steps=20, restarts=1, seed=0):
    r = np.random.default_rng(seed)
    best = X.copy(); best_loss = -np.inf*np.ones(len(X))
    for _ in range(restarts):
        Xa = X + r.uniform(-eps, eps, X.shape)
        for _ in range(steps):
            Xa = Xa + (2.5*eps/steps)*np.sign(input_grad(p, Xa, Y))
            Xa = X + np.clip(Xa - X, -eps, eps)
        z, _ = forward(p, Xa)
        l = np.abs(z)*np.where((z > 0) == (Y > 0.5), -1, 1)
        upd = l > best_loss
        best[upd] = Xa[upd]; best_loss[upd] = l[upd]
    return best

p_std = train(init(), X, Y)
acc = lambda p, Xe: np.mean((forward(p, Xe)[0] > 0) == (Y > 0.5))
print(f"temiz doğruluk: {acc(p_std, X):.3f}")

eps_grid = np.linspace(0, 0.5, 21)
acc_fgsm = [acc(p_std, fgsm(p_std, X, Y, e)) for e in eps_grid]
acc_pgd  = [acc(p_std, pgd(p_std, X, Y, e, restarts=3)) for e in eps_grid]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(eps_grid, acc_fgsm, "o-", lw=2, label="FGSM (1 adım)")
axes[0].plot(eps_grid, acc_pgd, "s-", lw=2, label="PGD (20 adım, 3 yeniden başlatma)")
axes[0].set_xlabel("eps"); axes[0].set_ylabel("saldırı altında doğruluk")
axes[0].set_title("Zayıf bir saldırı dayanıklılığı abartır"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

steps_list = [1, 2, 5, 10, 20, 50]
axes[1].plot(steps_list, [acc(p_std, pgd(p_std, X, Y, 0.25, s)) for s in steps_list], "o-", lw=2)
axes[1].set_xlabel("PGD adımı"); axes[1].set_ylabel("saldırı altında doğruluk (eps=0.25)")
axes[1].set_title("Saldırı güçlendikçe raporlanan dayanıklılık düşüyor"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("Kural: dayanıklılık iddiaları ancak çalıştırabildiğiniz en güçlü saldırıya karşı anlamlıdır.")


## 3. Düşmanca Eğitim ve Dayanıklılık–Doğruluk Takası

Standart savunma, saldırıya uğramış girdilerle eğitmektir — Madry'nin eyer noktası formülasyonu:

$$\min_\theta\ \mathbb{E}\Big[\max_{\lVert\delta\rVert\le\epsilon} \mathcal{L}(f_\theta(x+\delta), y)\Big].$$

Her adım tam bir PGD iç döngüsü gerektirdiğinden eğitim kabaca $k$ kat pahalıdır. Ve genel olarak
**temiz doğruluğu düşürür**: veride öngörücü ama kırılgan öznitelikler varsa dayanıklı sınıflandırıcı
ile doğru sınıflandırıcı gerçekten farklı fonksiyonlardır. Bir modelin, etiketle zayıf korelasyonlu
ve kolayca ters çevrilebilen bir özniteliği kullanmasına izin vardır; dayanıklılık bunu yasaklar ve o
bilgi kaybolur.


In [ ]:
p_adv = train(init(), X, Y, steps=800, adv_eps=0.25, pgd_steps=5)
p_std2 = train(init(), X, Y, steps=800)

eps_grid = np.linspace(0, 0.6, 16)
curves = {
    "standart eğitim":    [acc(p_std2, pgd(p_std2, X, Y, e, 20)) for e in eps_grid],
    "düşmanca eğitim": [acc(p_adv,  pgd(p_adv,  X, Y, e, 20)) for e in eps_grid],
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for name, c in curves.items():
    axes[0].plot(eps_grid, c, "o-", lw=2, label=name)
axes[0].set_xlabel("saldırı bütçesi eps"); axes[0].set_ylabel("doğruluk")
axes[0].set_title("Düşmanca eğitim dayanıklılık satın alır..."); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].bar(["standart", "düşmanca"], [curves["standart eğitim"][0], curves["düşmanca eğitim"][0]],
            color=["#6baed6", "#08519c"])
axes[1].set_ylim(0.8, 1.0); axes[1].set_ylabel("temiz doğruluk")
axes[1].set_title("...ve bedeli temiz doğruluktur")

# Neden: dayanıklı ve dayanıksız öznitelikler
n2 = 4000
r = np.random.default_rng(3)
y2 = r.integers(0, 2, n2)*2 - 1
robust_feat = y2*1.5 + r.normal(size=n2)*1.0                  # güçlü sinyal, ters çevirmesi zor
weak_feats  = y2[:, None]*0.12 + r.normal(size=(n2, 40))*0.12 # çok sayıda zayıf, kolayca ters çevrilebilen öznitelik
axes[2].hist(robust_feat[y2 > 0], bins=40, alpha=0.6, label="dayanıklı öznitelik | y=+1")
axes[2].hist(robust_feat[y2 < 0], bins=40, alpha=0.6, label="dayanıklı öznitelik | y=-1")
axes[2].hist(weak_feats[:, 0][y2 > 0], bins=40, alpha=0.6, label="dayanıksız öznitelik | y=+1")
axes[2].set_title("Dayanıksız öznitelikler öngörücü ama kırılgandır"); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

score = weak_feats.sum(1)
eps_w = 0.14                                                  # öznitelik başına bozulma
score_att = score - y2*eps_w*weak_feats.shape[1]              # her zayıf özniteliği yanlış yöne it
print(f"yalnızca {weak_feats.shape[1]} zayıf öznitelik {np.mean(np.sign(score)==y2):.3f} temiz doğruluk veriyor")
print(f"...ve her biri yalnızca {eps_w} kadar kaydırılınca {np.mean(np.sign(score_att)==y2):.3f}")
print("Bunları kullanan bir sınıflandırıcı doğrudur ama dayanıklı değildir; düşmanca eğitim bu takası yasaklar.")


## 4. Rastgeleleştirilmiş Yumuşatma ile Sertifikalı Dayanıklılık

Deneysel savunmalar kırılır ve yeniden kırılır; bir **sertifika** bu döngüyü bitirir.
Rastgeleleştirilmiş yumuşatma, herhangi bir taban sınıflandırıcı $f$'yi şuna çevirir:

$$g(x) = \arg\max_c\ \mathbb{P}_{\eta \sim \mathcal{N}(0,\sigma^2 I)}\big[f(x+\eta) = c\big],$$

ve garanti nettir: en olası sınıfın olasılığı $\underline{p_A}$, ikincininki $\overline{p_B}$ ise
$g$, yarıçapı

$$R = \frac{\sigma}{2}\big(\Phi^{-1}(\underline{p_A}) - \Phi^{-1}(\overline{p_B})\big)$$

olan bir $\ell_2$ yuvarı içinde sabittir. Takas açık ve kaçınılmazdır: büyük $\sigma$ daha büyük
yarıçapları sertifikalar ama taban sınıflandırıcının doğruluğunu bozar; bu yüzden sertifikalı
doğruluk eğrisinin her yarıçap için bir iç optimumu vardır.


In [ ]:
def certified_radius(pA, pB, sigma):
    pA, pB = np.clip(pA, 1e-6, 1-1e-6), np.clip(pB, 1e-6, 1-1e-6)
    return np.maximum(sigma/2*(norm.ppf(pA) - norm.ppf(pB)), 0)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
pA = np.linspace(0.5, 0.9999, 400)
for s, c in zip([0.12, 0.25, 0.5, 1.0], ["#c6dbef", "#6baed6", "#2171b5", "#08306b"]):
    axes[0].plot(pA, certified_radius(pA, 1-pA, s), lw=2, c=c, label=f"sigma={s}")
axes[0].set_xlabel("gürültü altında en olası sınıfın olasılığı"); axes[0].set_ylabel("sertifikalı L2 yarıçapı")
axes[0].set_title("Sertifikalar çok yüksek güven gerektirir")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Sertifikalı doğruluk-yarıçap: büyük sigma uzakta yardım eder, yakında zarar verir
radii = np.linspace(0, 1.6, 60)
for s, c in zip([0.12, 0.25, 0.5, 1.0], ["#c6dbef", "#6baed6", "#2171b5", "#08306b"]):
    base_acc = 0.95*np.exp(-(s/0.75)**2)                       # gürültü taban sınıflandırıcıyı bozar
    r_ = np.random.default_rng(0)
    pa = np.clip(r_.beta(2, 1, 4000)*base_acc + (1-base_acc)*0.5, 0.5, 0.9999)
    cr = certified_radius(pa, 1-pa, s)
    axes[1].plot(radii, [(cr >= t).mean()*base_acc for t in radii], lw=2, c=c, label=f"sigma={s}")
axes[1].set_xlabel("L2 yarıçapı"); axes[1].set_ylabel("sertifikalı doğruluk")
axes[1].set_title("Tek bir sigma her yerde üstün değil"); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

# Örnekleme maliyeti: sınır, pA üzerinde bir güven aralığı gerektirir
ns = np.logspace(2, 6, 60)
for pa_true, c in zip([0.9, 0.99, 0.999], ["#6baed6", "#2171b5", "#08306b"]):
    lower = pa_true - 1.96*np.sqrt(pa_true*(1-pa_true)/ns)
    axes[2].semilogx(ns, certified_radius(np.clip(lower, 0.5, 1-1e-6), 1-np.clip(lower, 0.5, 1-1e-6), 0.5),
                     lw=2, c=c, label=f"gerçek pA={pa_true}")
axes[2].set_xlabel("girdi başına Monte-Carlo örneği"); axes[2].set_ylabel("elde edilen sertifikalı yarıçap")
axes[2].set_title("Sertifikasyon çıkarım zamanında pahalıdır")
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()
print("Sınırın biçimine dikkat: pA -> 1 sınırsız yarıçap verir; 10^5 örneğin tipik olmasının nedeni budur.")


## 5. Sahte Korelasyonlar ve En Kötü Grup Doğruluğu

Düşmanca saldırıdan çok daha yaygın bir başarısızlık: eğitimde etiketle korelasyonlu ama nedensel
ilgisi olmayan bir öznitelik. Ortalama doğruluk bunu tamamen gizler; çünkü korelasyonun bozulduğu
azınlık grupları, tanımı gereği verinin küçük bir kısmıdır.

Teşhis **en kötü grup doğruluğudur** ve standart çareler grup-dengeli yeniden ağırlıklandırma ile
ortalama yerine maksimum grup kaybını minimize eden **grup DRO**'dur:

$$\min_\theta\ \max_{g \in \mathcal{G}}\ \mathbb{E}_{(x,y)\sim P_g}\big[\mathcal{L}(f_\theta(x), y)\big].$$


In [ ]:
# İki öznitelik: 'çekirdek' (nedensel, daha zor) ve 'sahte' (kolay, zamanın %95'inde korelasyonlu)
def make_data(n, corr, seed=0):
    r = np.random.default_rng(seed)
    y = r.integers(0, 2, n)*2 - 1
    s = np.where(r.random(n) < corr, y, -y)                  # sahte özellik
    x_core = y*1.0 + r.normal(size=n)*1.2                    # zayıf ama nedensel
    x_spur = s*2.0 + r.normal(size=n)*0.3                    # güçlü ama sahte
    return np.c_[x_core, x_spur], y, s

Xtr, ytr, str_ = make_data(4000, 0.95, 0)
Xte, yte, ste  = make_data(4000, 0.50, 1)                    # test: korelasyon bozuldu

def fit(X, y, weights=None, steps=2000, lr=0.5):
    w, b = np.zeros(X.shape[1]), 0.0
    wt = np.ones(len(X)) if weights is None else weights
    wt = wt/wt.sum()
    for _ in range(steps):
        p = sigmoid(X@w + b)
        g = (p - (y > 0))*wt
        w -= lr*(X.T@g); b -= lr*g.sum()
    return w, b

def group_acc(w, b, X, y, s):
    out = {}
    for gy in [-1, 1]:
        for gs in [-1, 1]:
            m = (y == gy) & (s == gs)
            out[f"y={gy:+d},s={gs:+d}"] = float(np.mean((np.sign(X[m]@w + b) == y[m]))) if m.sum() else np.nan
    return out

w_erm, b_erm = fit(Xtr, ytr)

groups = [(gy, gs) for gy in [-1, 1] for gs in [-1, 1]]
counts = np.array([np.sum((ytr == gy) & (str_ == gs)) for gy, gs in groups])
w_bal = fit(Xtr, ytr, weights=np.array([1/counts[groups.index((y_, s_))] for y_, s_ in zip(ytr, str_)]))

# Grup DRO: ağırlığı, o an en yüksek kayba sahip gruba doğru kaydır
q = np.ones(4)/4
w_dro, b_dro = np.zeros(2), 0.0
for _ in range(2000):
    losses = []
    for gi, (gy, gs) in enumerate(groups):
        m = (ytr == gy) & (str_ == gs)
        p = sigmoid(Xtr[m]@w_dro + b_dro)
        losses.append(-np.mean((ytr[m] > 0)*np.log(p+1e-9) + (ytr[m] <= 0)*np.log(1-p+1e-9)))
    q = q*np.exp(0.1*np.array(losses)); q /= q.sum()
    gw, gb = np.zeros(2), 0.0
    for gi, (gy, gs) in enumerate(groups):
        m = (ytr == gy) & (str_ == gs)
        p = sigmoid(Xtr[m]@w_dro + b_dro)
        e = (p - (ytr[m] > 0))/m.sum()
        gw += q[gi]*(Xtr[m].T@e); gb += q[gi]*e.sum()
    w_dro -= 0.5*gw; b_dro -= 0.5*gb

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
names = ["ERM", "grup-dengeli", "grup DRO"]
models = [(w_erm, b_erm), w_bal, (w_dro, b_dro)]
avg, worst = [], []
for (w, b) in models:
    ga = group_acc(w, b, Xte, yte, ste)
    avg.append(np.mean((np.sign(Xte@w + b) == yte)))
    worst.append(min(ga.values()))
x = np.arange(3); wd = 0.35
axes[0].bar(x-wd/2, avg, wd, label="ortalama doğruluk")
axes[0].bar(x+wd/2, worst, wd, label="en kötü grup doğruluğu")
axes[0].set_xticks(x); axes[0].set_xticklabels(names); axes[0].set_ylim(0, 1)
axes[0].legend(fontsize=9); axes[0].set_title("Ortalama doğruluk başarısızlığı gizliyor")

for ax, (w, b), nm in zip(axes[1:], [models[0], models[2]], ["ERM", "group DRO"]):
    ax.scatter(Xte[ste == yte, 0], Xte[ste == yte, 1], s=6, alpha=0.4, label="çoğunluk grupları")
    ax.scatter(Xte[ste != yte, 0], Xte[ste != yte, 1], s=6, alpha=0.6, c="crimson", label="azınlık grupları")
    xs = np.linspace(-5, 5, 10)
    ax.plot(xs, -(w[0]*xs + b)/(w[1]+1e-9), "k--", lw=2)
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
    ax.set_xlabel("çekirdek öznitelik"); ax.set_ylabel("sahte öznitelik")
    ax.set_title(f"{nm}: w = {np.round(w, 2)}"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

for nm, (w, b) in zip(names, models):
    print(f"{nm:16s} ağırlıklar {np.round(w,2)}  ortalama {np.mean((np.sign(Xte@w+b)==yte)):.3f}  "
          f"en kötü grup {min(group_acc(w,b,Xte,yte,ste).values()):.3f}")


## 6. Ortak Değişken Kayması, Tespit ve OOD

Yalnızca $p(x)$ değişip $p(y \mid x)$ değişmiyorsa buna **ortak değişken kayması** (covariate shift)
denir ve ilkesel olarak $w(x) = p_{\text{test}}(x)/p_{\text{eğitim}}(x)$ önem ağırlıklandırmasıyla
düzeltilebilir. Pratikte destekler farklılaştığında ağırlıkların varyansı devasa olur ve **etkin
örneklem büyüklüğü** $(\sum w)^2/\sum w^2$ çöker — yeniden ağırlıklandırılmış herhangi bir kestirime
güvenmeden önce hesaplanması gereken iyi bir teşhistir.

Tek bir girdinin dağılım dışı (OOD) olduğunu tespit etmek ise ayrı bir problemdir. Artan maliyet ve
genellikle artan kalite sırasıyla üç yaygın skor:

- **Maksimum softmax olasılığı** — zayıf bir temel yöntem; ağlar dağılım dışında kendinden emin
  biçimde yanılır.
- **Enerji** $-\log\sum_c e^{z_c}$ — tüm logit vektörünü kullanır, yoğunluğa daha iyi kalibredir.
- **Öznitelik uzayında Mahalanobis uzaklığı** — sınıf-koşullu öznitelik dağılımını modeller.


In [ ]:
rng = np.random.default_rng(2)
d, K = 8, 5

# Artan ortak değişken kayması altında önem ağırlıkları ve etkin örneklem büyüklüğü
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
shifts = np.linspace(0, 2.5, 25)
ess = []
for sft in shifts:
    Xt = rng.normal(size=(3000, d)) + sft                     # test dağılımı: kaydırılmış ortalama
    logw = -0.5*(Xt**2).sum(1) + 0.5*((Xt-sft)**2).sum(1)     # p_eğitim(x)/p_test(x) ters orandır
    w = np.exp(logw - logw.max())
    ess.append((w.sum()**2)/(w**2).sum()/len(w))
axes[0].plot(shifts, ess, "o-", lw=2)
axes[0].set_xlabel("eğitim ile test arasındaki ortalama kayma"); axes[0].set_ylabel("etkin örneklem büyüklüğü oranı")
axes[0].set_title("Önem ağırlıklandırma hızla bozuluyor"); axes[0].grid(alpha=0.3); axes[0].set_ylim(0, 1.05)

# Eğitilmiş 5 sınıflı bir sınıflandırıcı; OOD noktalar her sınıftan uzakta
MU = np.zeros((K, d))
for k in range(K): MU[k, k] = 3.0
y = rng.integers(0, K, 3000)
Xin = MU[y] + rng.normal(size=(3000, d))
u = rng.normal(size=(3000, d)); u /= np.linalg.norm(u, axis=1, keepdims=True)
Xout = 8*u + rng.normal(size=(3000, d))

W, b = np.zeros((d, K)), np.zeros(K)
for _ in range(2000):
    z = Xin@W + b
    p = np.exp(z - z.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
    g = (p - np.eye(K)[y])/len(Xin)
    W -= 0.5*(Xin.T@g); b -= 0.5*g.sum(0)
print(f"dağılım içi doğruluk: {np.mean((Xin@W+b).argmax(1) == y):.3f}")

msp    = lambda z: np.exp(z-z.max(1, keepdims=True)).max(1)/np.exp(z-z.max(1, keepdims=True)).sum(1)
energy = lambda z: -(z.max(1) + np.log(np.exp(z-z.max(1, keepdims=True)).sum(1)))
Sinv   = np.linalg.inv(np.cov(Xin.T))
maha   = lambda X: np.min([np.einsum("ij,jk,ik->i", X-MU[k], Sinv, X-MU[k]) for k in range(K)], axis=0)

zi, zo = Xin@W + b, Xout@W + b
scores = {"maks softmax": (-msp(zi), -msp(zo)),
          "enerji":      (energy(zi), energy(zo)),
          "Mahalanobis": (maha(Xin), maha(Xout))}

def auroc(s_in, s_out):
    allv = np.concatenate([s_in, s_out])
    r = np.argsort(np.argsort(allv)) + 1
    n1, n2 = len(s_in), len(s_out)
    return (r[n1:].sum() - n2*(n2+1)/2)/(n1*n2)

axes[1].hist(msp(zi), bins=50, alpha=0.6, density=True, label="dağılım içi")
axes[1].hist(msp(zo), bins=50, alpha=0.6, density=True, label="dağılım dışı")
axes[1].set_xlabel("maksimum softmax olasılığı"); axes[1].set_ylabel("yoğunluk")
axes[1].set_title("Güven ikisini ayırmıyor"); axes[1].legend(fontsize=9)

au = [auroc(*v) for v in scores.values()]
axes[2].barh(list(scores), au, color=["#c6dbef", "#6baed6", "#08519c"])
axes[2].axvline(0.5, c="k", ls="--", lw=1)
axes[2].set_xlim(0, 1.0); axes[2].set_xlabel("AUROC (içeri - dışarı)")
axes[2].set_title("OOD tespit kalitesi")
plt.tight_layout(); plt.show()
for name, a_ in zip(scores, au):
    print(f"{name:14s} AUROC {a_:.3f}")


Sonuç üzerinde durmaya değer. Bir logit, girdinin *sınırsız* bir fonksiyonudur; dolayısıyla bir sınıf
yönünde veriden uzaklaşmak modeli daha az değil **daha çok** kendinden emin yapar. Maksimum softmax
burada bu yüzden şansa yakındır ve enerji fiilen ters korelasyonludur — uzaktaki noktaları en
"dağılım içi" noktalar olarak sıralar. Yalnızca öznitelik **yoğunluğunu** modelleyen skor, yani
Mahalanobis, işe yarar.

Genel ders: güven tabanlı OOD tespiti, ayırt edici bir modele üretici bir soru sorar. Gerçek ağlarda
bazen işe yaraması, yönteminin sağlam olmasından değil özniteliklerinin doyuma ulaşmasındandır.


## 7. Özet

| Kavram | Açıklama |
|---|---|
| **Doğrusal açıklama** | $\epsilon\lVert w\rVert_1$ boyutla büyür; piksel başına küçük değişimler yeter |
| **FGSM / PGD** | Tek işaretli adım ile rastgele başlangıçlı yinelemeli izdüşümlü adımlar |
| **Değerlendirme disiplini** | Sonucu en güçlü saldırıya karşı raporlayın; zayıf saldırı dayanıklılık uydurur |
| **Düşmanca eğitim** | Eyer noktası amacı; $k$ kat maliyet ve daha düşük temiz doğruluk |
| **Dayanıksız öznitelikler** | Öngörücü ama kırılgan; dayanıklılık bunları bilerek atar |
| **Rastgeleleştirilmiş yumuşatma** | $R = \frac{\sigma}{2}(\Phi^{-1}(p_A)-\Phi^{-1}(p_B))$; sertifikalı ama pahalı |
| **Sahte korelasyon** | Ortalama doğruluk azınlık grubu başarısızlığını gizler |
| **Grup DRO** | Ortalama yerine en kötü grup kaybını minimize eder |
| **Ortak değişken kayması** | Önem ağırlıklandırma geçerlidir ama etkin örneklem büyüklüğü çöker |
| **OOD skorları** | Maks softmax < enerji < Mahalanobis; maliyette ve genellikle kalitede |

**Sonraki Defter →** Ölçekte Verimli Eğitim ve Çıkarım
